# Helix AI Modernizer Training (Colab)

This notebook fine-tunes a practical model for the project goal: legacy Python modernization.

Recommended runtime: **GPU** on Google Colab.

Model: `unsloth/Qwen2.5-Coder-1.5B-Instruct`


In [ ]:
!pip -q install --upgrade unsloth datasets peft trl accelerate sentencepiece transformers


In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

DATA_PATH = "/content/training_mixture.jsonl"
if "training_mixture.jsonl" not in uploaded:
    raise RuntimeError("Upload ml/data/training_mixture.jsonl from the project.")


In [ ]:
import json
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_DIR = "/content/nebula-modernizer-qwen25-1.5b"
MAX_SEQ_LENGTH = 2048

PROMPT_TEMPLATE = """You are Helix AI, a legacy Python modernization model.
Convert old Python code to current Python while preserving behavior.
Return only the final migrated code.

### Instruction:
{instruction}

### Input:
{input_code}

### Response:
{output_code}"""

records = []
with open(DATA_PATH, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if line:
            records.append(json.loads(line))

dataset = Dataset.from_list(records).shuffle(seed=42)

def format_batch(batch):
    texts = []
    for instruction, input_code, output_code in zip(batch["instruction"], batch["input"], batch["output"]):
        texts.append(PROMPT_TEMPLATE.format(instruction=instruction, input_code=input_code, output_code=output_code))
    return {"text": texts}

dataset = dataset.map(format_batch, batched=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        num_train_epochs=2,
        warmup_ratio=0.05,
        logging_steps=5,
        save_strategy="epoch",
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        report_to=[],
    ),
)

trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved", OUTPUT_DIR)


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("nebula-modernizer-qwen25-1.5b", "zip", root_dir=OUTPUT_DIR)
print("archive", archive)
files.download(archive)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, OUTPUT_DIR)

prompt = """You are Helix AI, a legacy Python modernization model.
Convert old Python code to current Python while preserving behavior.
Return only the final migrated code.

### Instruction:
Modernize this legacy Python code to current Python while preserving behavior.

### Input:
for i in xrange(3):\n    print i\n

### Response:
"""

inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded.split("### Response:")[-1].strip())
